In [ ]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) return false;

In [ ]:
import dataset, plots
import pandas as pd

from config import INTERIM_DATA_DIR

sp500_components = dataset.SP500.load_historical()

## Połączenie danych z Yahoo Finance i EODHD

Dane z obu źródeł są przechowywane w formacie kolumnowym — każdy plik CSV odpowiada jednej zmiennej (np. `Close.csv`, `Adj_Close.csv`), a kolumny oznaczają tickery. Poniżej definiujemy dwie funkcje:

- **`merge_price_data`** — łączy dwa słowniki ramek danych (Yahoo Finance i EODHD) w jeden słownik. Dla tickerów obecnych w obu źródłach preferowane są dane z EODHD.
- **`save_merged_data`** — zapisuje scalony słownik ramek do katalogu w tym samym formacie co pliki źródłowe w `data/external/`.

### Wczytanie danych źródłowych

In [ ]:
yf_data: dict[str, pd.DataFrame] = dataset.YahooFinance.load()
for column_name, frame in yf_data.items():
    if frame.empty:
        print(f"{column_name}: empty")
    else:
        print(f"{column_name}: {frame.shape}, {frame.index.min().date()} -> {frame.index.max().date()}")

In [ ]:
eodhd_data: dict[str, pd.DataFrame] = dataset.EODHD.load()
for column_name, frame in eodhd_data.items():
    if frame.empty:
        print(f"{column_name}: empty")
    else:
        print(f"{column_name}: {frame.shape}, {frame.index.min().date()} -> {frame.index.max().date()}")

### Scalanie i zapis

In [ ]:
merged_data = dataset.merge_price_data(yf_data, eodhd_data)

for column_name, frame in merged_data.items():
    if frame.empty:
        print(f"{column_name}: empty")
    else:
        print(f"{column_name}: {frame.shape}, {frame.index.min().date()} -> {frame.index.max().date()}")

In [ ]:
output_path = INTERIM_DATA_DIR / "merged"
dataset.save_merged_data(merged_data, output_path)

### Pokrycie danych

In [ ]:
coverage_df = plots.coverage_over_time(merged_data, sp500_components)
plots.summarize_df(coverage_df)

In [ ]:
plots.plot_missing_data_reasons("Merged")